In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Processed_527_datapoints.csv"
ARTIFACTS = PROJECT_ROOT / "artifacts"
TARGET = "CO2 uptake (mmol/g)"
df = pd.read_csv(RAW_PATH)
df.shape

(527, 11)

In [2]:
df = df.drop_duplicates().reset_index(drop=True)
df.insert(0, "row_id", range(len(df)))
df.shape

(526, 12)

In [3]:
dev_ids, test_ids = train_test_split(df["row_id"], test_size=0.20, random_state=42)
train_ids, val_ids = train_test_split(dev_ids, test_size=0.20, random_state=42)

df["split"] = "train"
df.loc[df["row_id"].isin(val_ids), "split"] = "validation"
df.loc[df["row_id"].isin(test_ids), "split"] = "test"

df["split"].value_counts()

split
train         336
test          106
validation     84
Name: count, dtype: int64

In [4]:
assert not set(train_ids) & set(val_ids)
assert not set(train_ids) & set(test_ids)
assert not set(val_ids) & set(test_ids)
print("Clean: no row is in two groups.")

Clean: no row is in two groups.


In [6]:
ARTIFACTS.mkdir(exist_ok=True)
df.to_csv(ARTIFACTS / "clean_data.csv", index=False)

split_config = {
    "random_state": 42,
    "duplicates_removed": 1,
    "counts": df["split"].value_counts().to_dict(),
}
(ARTIFACTS / "split_config.json").write_text(json.dumps(split_config, indent=2))
split_config

{'random_state': 42,
 'duplicates_removed': 1,
 'counts': {'train': 336, 'test': 106, 'validation': 84}}

In [7]:
FEATURES = [c for c in df.columns if c not in ("row_id", "split", TARGET)]

train = df[df["split"] == "train"]
val   = df[df["split"] == "validation"]
test  = df[df["split"] == "test"]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(len(FEATURES), "features")
pd.DataFrame({
    "train": X_train.isna().sum(),
    "validation": X_val.isna().sum(),
    "test": X_test.isna().sum(),
}).query("train > 0 or validation > 0 or test > 0")

10 features


,train,validation,test
Total Pore Volume(cm3/g),19,6,3
Micropore Volume (cm3/g),142,34,34


In [8]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)                       # LEARN medians from training only

medians = dict(zip(FEATURES, imputer.statistics_))
print("Learned medians:", {k: round(v, 3) for k, v in medians.items()})

X_train_filled = imputer.transform(X_train)   # APPLY to every group
X_val_filled   = imputer.transform(X_val)
X_test_filled  = imputer.transform(X_test)

print("Blanks left:", np.isnan(X_train_filled).sum(), np.isnan(X_val_filled).sum(), np.isnan(X_test_filled).sum())

Learned medians: {'Surface Area (m2/g)': np.float64(1423.05), 'Total Pore Volume(cm3/g)': np.float64(0.74), 'Micropore Volume (cm3/g)': np.float64(0.48), 'C (%)': np.float64(78.855), 'H (%)': np.float64(1.5), 'N (%)': np.float64(2.26), 'O (%)': np.float64(15.05), 'S (%)': np.float64(0.0), 'Temp (°C)': np.float64(25.0), 'Pressure (bar)': np.float64(1.0)}
Blanks left: 0 0 0


In [9]:
mpv_col = FEATURES.index("Micropore Volume (cm3/g)")
same = (X_train_filled[:, mpv_col] == medians["Micropore Volume (cm3/g)"]).sum()
print(f"{same} of {len(X_train_filled)} training rows now have micropore volume = exactly 0.48")

157 of 336 training rows now have micropore volume = exactly 0.48


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ridge_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
ridge_prep.fit(X_train)

train_scaled = pd.DataFrame(ridge_prep.transform(X_train), columns=FEATURES)
val_scaled   = pd.DataFrame(ridge_prep.transform(X_val),   columns=FEATURES)

pd.DataFrame({
    "train mean": train_scaled.mean().round(2),
    "train std":  train_scaled.std(ddof=0).round(2),
    "val mean":   val_scaled.mean().round(2),
}).head(5)

,train mean,train std,val mean
Surface Area (m2/g),0.0,1.0,-0.17
Total Pore Volume(cm3/g),0.0,1.0,-0.03
Micropore Volume (cm3/g),-0.0,1.0,-0.15
C (%),-0.0,1.0,-0.04
H (%),-0.0,1.0,0.10
